# sic: OpenICU → YAIB wide

This is the complete dataset workflow. It writes the all-concept OpenICU wide table below the OpenICU workspace `yaib/sic/` directory. For RICU validation it additionally creates OpenICU and RICU 7-day wide tables and comparison reports.


In [ ]:
from pathlib import Path
import os
import subprocess

from openicu_yaib import (
    build_and_write_yaib_wide_for_dataset,
    compare_openicu_wide_to_ricu_for_dataset,
    concept_root_from_output,
    resolve_openicu_workspace,
    write_all_concepts_wide,
)

DATASET = "sic"
MAX_HOURS = 7 * 24

# Point this at either the OpenICU project root, its workspace directory,
# the concept directory, or a separate output directory with concept/ below it.
OPENICU_OUTPUT = Path.home() / "output" / "OpenICU.example" / "output" / "project" / "workspace"
WORKSPACE = resolve_openicu_workspace(OPENICU_OUTPUT)
CONCEPT_ROOT = concept_root_from_output(OPENICU_OUTPUT)
DATASET_OUTPUT = WORKSPACE / "yaib" / DATASET
DATASET_OUTPUT.mkdir(parents=True, exist_ok=True)

# Optional override if automatic raw stay-table discovery is not sufficient:
STAYS_PATH = None
RICU_CONCEPT_DICT = Path.home() / "workspace" / "ricu" / "inst" / "extdata" / "config" / "concept-dict.json"


In [ ]:
# 1) Full OpenICU YAIB-wide table across every available concept.
all_result = write_all_concepts_wide(
    dataset=DATASET,
    openicu_output=OPENICU_OUTPUT,
    concept_root=CONCEPT_ROOT,
    stays_path=STAYS_PATH,
    max_hours=None,
)
all_result


In [ ]:
# 2) Export the matching RICU reference files.
REPO_ROOT = Path("/home/q039tl/workspace/openicu-yaib")

env = os.environ.copy()
env["RICU_OUT_DIR"] = str(DATASET_OUTPUT)
env["RICU_DATA_PATH"] = "/home/q039tl/ricu_data"
env["RICU_SRC_LOAD"] = (
    "mimic,mimic_demo,eicu,eicu_demo,hirid,aumc,miiv,sic"
)
env.pop("RICU_CONFIG_PATH", None)
env["R_ENVIRON_USER"] = "/dev/null"

subprocess.run(
    [
        "Rscript",
        "--no-environ",
        "scripts/datasets/export_ricu_sic.R",
    ],
    cwd=REPO_ROOT,
    check=True,
    env=env,
)


In [ ]:
# 3) OpenICU YAIB-wide table for exactly the first 7 days.
openicu_7d = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    version="1.0.0",
)
openicu_7d


In [ ]:
# 4) Compare only the 7-day OpenICU and RICU wide representations.
comparison = compare_openicu_wide_to_ricu_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=DATASET_OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=STAYS_PATH,
    ricu_concept_dict=RICU_CONCEPT_DICT,
)
comparison.as_dict()
